# Lab 13 — Multi-agent RAG from scratch

Compose Path 02's retrieval pipeline with Path 03's coordination
patterns. A retriever-worker wraps Lab 06-08's pipeline; a supervisor
decides when retrieval is needed; a synthesizer composes the final
answer with citation preservation.

Headline path: 2-4 LLM calls per query, ~3-6 seconds wall-clock.
Optional stretch sections add Lab 11's critic (groundedness) and Lab
12's planner (parallel retrievals for compound questions).

> ⏱ Run time: 130-160 min including reading.
> 📖 Read [`concepts/multi-agent/multi-agent-rag.md`](../../concepts/multi-agent/multi-agent-rag.md)
> and [`concepts/multi-agent/retriever-as-worker.md`](../../concepts/multi-agent/retriever-as-worker.md)
> first. The lab references the four failure modes and the four
> retrieval-decision rules directly.

## Step 0: Setup

Same provider-agnostic setup as Labs 10/11/12. The corpus lives in
Lab 06's directory; this lab loads chunks from there. If Lab 08's
contextual cache exists at `../08-contextual-retrieval-and-query-rewriting/context_cache.json`,
we'll use the v3 pipeline; otherwise we fall back to v2.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        REPO_ROOT = parent
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

# Locate the corpus (Lab 06's directory)
CORPUS_DIR = REPO_ROOT / "labs/06-agentic-rag-from-scratch/corpus"
assert CORPUS_DIR.exists(), f"Corpus not found at {CORPUS_DIR}; check Lab 06 setup"
print(f"Corpus dir: {CORPUS_DIR}")
print(f"  documents: {sum(1 for f in CORPUS_DIR.glob('*.md'))}")

# Optionally locate Lab 08's context cache
CONTEXT_CACHE_PATH = REPO_ROOT / "labs/08-contextual-retrieval-and-query-rewriting/context_cache.json"
HAS_CONTEXT_CACHE = CONTEXT_CACHE_PATH.exists()
print(f"Context cache (Lab 08): {'found — will use v3 pipeline' if HAS_CONTEXT_CACHE else 'not found — will use v2 pipeline'}")


**Sample output:**

```
Using openai / gpt-4o-mini
Corpus dir: /path/to/repo/labs/06-agentic-rag-from-scratch/corpus
  documents: 8
Context cache (Lab 08): not found — will use v2 pipeline
```

If you completed Lab 08 in the parent lab directory, you'll see the v3 message instead.

## Step 1: Compact recap

Lab 13 reuses:

- **From Lab 10/11/12**: `chat_with_tools` provider-agnostic chat client, `_action_hash` dedup, `StrictModel` Pydantic base, structured-error envelope pattern, the supervisor-worker pattern itself.
- **From Lab 06**: the corpus chunker (`TARGET_TOKENS=160`, `OVERLAP_TOKENS=32`) and chunk-loading discipline.
- **From Lab 07**: the v2 retrieval pipeline (dense + BM25 + RRF + cross-encoder rerank).
- **From Lab 08** (optionally): the v3 pipeline (contextual augmentation + query rewriting).

What Lab 13 adds: a retriever-worker that wraps the pipeline; a synthesizer-worker that composes from chunks with citation preservation; a supervisor that decides when retrieval is needed via the four retrieval-decision rules.

We restate the chat client + helpers inline; the retrieval pipeline gets its own steps.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
    temperature: float = 0,
) -> AssistantMessage:
    """Provider-agnostic chat client. Same as Lab 10/11/12."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=2048, temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
               for b in resp.content if getattr(b, "type", None) == "tool_use"]
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    """Lab 03/10/11/12 dedup helper."""
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    """Lab 02 pattern: extra='forbid'."""
    model_config = ConfigDict(extra="forbid")


## Step 2: Build chunks from Lab 06's corpus

Same chunker as Lab 06 — `TARGET_TOKENS=160`, `OVERLAP_TOKENS=32`. We
re-implement inline rather than importing from Lab 06's notebook
(notebooks aren't importable modules). The chunk IDs match what Lab 06
produces so Lab 08's cache (if present) lines up.

In [ ]:
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    """Word count divided by ~0.75 — close enough for chunking decisions."""
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    parts = re.split(r"\n\s*\n", text)
    return [p.strip() for p in parts if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def chunk_text(text: str, source: str, title: str) -> list[dict]:
    """Recursive splitting with overlap. Returns list of chunk dicts."""
    chunks: list[dict] = []
    paragraphs = split_at_paragraphs(text)

    current = ""
    chunk_idx = 0

    def emit_chunk(content: str) -> None:
        nonlocal chunk_idx
        if not content.strip():
            return
        chunk_id = f"{pathlib.Path(source).stem}_{chunk_idx:03d}"
        chunks.append({
            "chunk_id": chunk_id,
            "text": content.strip(),
            "source": source,
            "title": title,
        })
        chunk_idx += 1

    for para in paragraphs:
        candidate = (current + "\n\n" + para) if current else para
        if approx_tokens(candidate) <= TARGET_TOKENS:
            current = candidate
            continue
        # Emit current, start new with overlap
        if current:
            emit_chunk(current)
            tail_words = current.split()[-OVERLAP_TOKENS:]
            current = " ".join(tail_words) + "\n\n" + para
        else:
            # Single paragraph too big; split by sentences
            sentences = split_at_sentences(para)
            buf = ""
            for s in sentences:
                cand = (buf + " " + s) if buf else s
                if approx_tokens(cand) <= TARGET_TOKENS:
                    buf = cand
                else:
                    if buf:
                        emit_chunk(buf)
                    buf = s
            current = buf
    if current:
        emit_chunk(current)

    return chunks


def load_all_chunks(corpus_dir: pathlib.Path) -> list[dict]:
    chunks: list[dict] = []
    for md_file in sorted(corpus_dir.glob("*.md")):
        if md_file.name == "README.md":
            continue
        text = md_file.read_text()
        # Extract title from first H1 if present
        title_match = re.search(r"^#\s+(.+)$", text, re.M)
        title = title_match.group(1).strip() if title_match else md_file.stem
        chunks.extend(chunk_text(text, source=md_file.name, title=title))
    return chunks


ALL_CHUNKS = load_all_chunks(CORPUS_DIR)
print(f"Loaded {len(ALL_CHUNKS)} chunks from {len(list(CORPUS_DIR.glob('*.md'))) - 1} documents")
print(f"First chunk: id={ALL_CHUNKS[0]['chunk_id']}, source={ALL_CHUNKS[0]['source']}")
print(f"First chunk text (first 200 chars): {ALL_CHUNKS[0]['text'][:200]}...")


**Sample output:**

```
Loaded 21 chunks from 8 documents
First chunk: id=01-agent-loop_000, source=01-agent-loop.md
First chunk text (first 200 chars): # The Agent Loop\n\nAn agent loop is the simplest possible structure...
```

If Lab 06/07/08 have already run with the same parameters, these chunk IDs match the IDs the context cache uses (if present).

## Step 3: Build the retrieval pipeline (v2 inline; v3 if cache exists)

We rebuild Lab 07's four-stage pipeline inline: dense embeddings (MiniLM), BM25, RRF combiner, cross-encoder rerank. If Lab 08's context cache is available, we add contextual augmentation as a preprocessing step on the indexed text.

This is **the same retrieval pipeline as Labs 06-08, used unchanged**. Lab 13's job is to wrap it as a worker, not modify it. If you've completed Labs 06-08, you can skim this step — the retrieval logic isn't new.

In [ ]:
# Suppress sentence-transformers' verbose imports
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


# Load models — first run downloads ~100MB; subsequent runs are cached
print("Loading dense embedder (all-MiniLM-L6-v2)...")
DENSE_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Loading cross-encoder reranker (ms-marco-MiniLM-L-6-v2)...")
CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Models loaded.")


# Apply contextual augmentation if Lab 08's cache is available
CONTEXT_CACHE: dict[str, str] = {}
if HAS_CONTEXT_CACHE:
    try:
        CONTEXT_CACHE = json.loads(CONTEXT_CACHE_PATH.read_text())
        cache_chunks = set(CONTEXT_CACHE.keys())
        our_chunks = {c["chunk_id"] for c in ALL_CHUNKS}
        overlap = cache_chunks & our_chunks
        coverage = len(overlap) / len(our_chunks) if our_chunks else 0
        if coverage >= 0.9:
            print(f"Context cache covers {coverage:.0%} of our chunks; using v3 pipeline")
        else:
            print(f"WARNING: context cache covers only {coverage:.0%} of our chunks; "
                  "chunk parameters may differ. Falling back to v2.")
            CONTEXT_CACHE = {}
    except Exception as e:
        print(f"WARNING: could not load context cache ({e}); falling back to v2")
        CONTEXT_CACHE = {}

PIPELINE = "v3-contextual" if CONTEXT_CACHE else "v2"
print(f"Pipeline: {PIPELINE}")


def _indexed_text(chunk: dict) -> str:
    """Return the text that gets indexed.

    For v3: contextual prefix + original chunk text.
    For v2: just the original chunk text.

    The agent always sees the ORIGINAL chunk text in retrieval results —
    contextual augmentation is indexing-only (same as Lab 08).
    """
    if CONTEXT_CACHE and chunk["chunk_id"] in CONTEXT_CACHE:
        return CONTEXT_CACHE[chunk["chunk_id"]] + "\n\n" + chunk["text"]
    return chunk["text"]


# Build dense index
print("Embedding chunks...")
INDEXED_TEXTS = [_indexed_text(c) for c in ALL_CHUNKS]
DENSE_INDEX = DENSE_MODEL.encode(INDEXED_TEXTS, convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False)
print(f"Dense index: {DENSE_INDEX.shape}")

# Build BM25 index
print("Building BM25 index...")
TOKENIZED_CHUNKS = [t.lower().split() for t in INDEXED_TEXTS]
BM25 = BM25Okapi(TOKENIZED_CHUNKS)
print(f"BM25 index: {len(TOKENIZED_CHUNKS)} chunks")


Now the four-stage retrieval function — same shape as Lab 07's `search_corpus_v2`:

In [ ]:
def _rrf_merge(dense_ranks: list[int], bm25_ranks: list[int], k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion. Returns chunk indices sorted by fused score."""
    scores: dict[int, float] = {}
    for rank, idx in enumerate(dense_ranks, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    for rank, idx in enumerate(bm25_ranks, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    return [idx for idx, _ in sorted(scores.items(), key=lambda kv: -kv[1])]


def retrieve_chunks(query: str, top_k: int = 5, candidate_k: int = 30) -> list[dict]:
    """The full Lab 07 v2 retrieval pipeline.

    Stages:
    1. Dense search (top candidate_k)
    2. BM25 search (top candidate_k)
    3. RRF merge → top candidate_k unique
    4. Cross-encoder rerank → top_k

    Returns chunks ranked by final rerank score.
    """
    # 1. Dense
    query_vec = DENSE_MODEL.encode([query], convert_to_numpy=True,
                                    normalize_embeddings=True, show_progress_bar=False)[0]
    dense_scores = DENSE_INDEX @ query_vec  # cosine since normalized
    dense_ranks = np.argsort(-dense_scores)[:candidate_k].tolist()

    # 2. BM25
    bm25_scores = BM25.get_scores(query.lower().split())
    bm25_ranks = np.argsort(-bm25_scores)[:candidate_k].tolist()

    # 3. RRF
    merged_ranks = _rrf_merge(dense_ranks, bm25_ranks)[:candidate_k]

    # 4. Cross-encoder rerank
    pairs = [(query, INDEXED_TEXTS[idx]) for idx in merged_ranks]
    rerank_scores = CROSS_ENCODER.predict(pairs, show_progress_bar=False)
    reranked = sorted(zip(merged_ranks, rerank_scores, strict=False), key=lambda kv: -kv[1])

    # Build result chunks with the ORIGINAL chunk text (not the indexed text)
    results = []
    for idx, score in reranked[:top_k]:
        c = ALL_CHUNKS[idx]
        results.append({
            "id": c["chunk_id"],
            "text": c["text"],
            "source": c["source"],
            "title": c["title"],
            "score": float(score),
        })
    return results


# Quick sanity check
sample = retrieve_chunks("what is an agent loop?", top_k=3)
print(f"Retrieved {len(sample)} chunks for 'what is an agent loop?':")
for s in sample:
    print(f"  [{s['id']}] (score={s['score']:.3f}) {s['title']}: {s['text'][:80]}...")


**Sample output:**

```
Retrieved 3 chunks for 'what is an agent loop?':
  [01-agent-loop_000] (score=8.7) The Agent Loop: An agent loop is the simplest possible structure...
  [01-agent-loop_001] (score=5.3) The Agent Loop: Each iteration of the loop has three parts...
  [03-react-pattern_000] (score=2.1) The ReAct Pattern: ReAct generalizes the agent loop...
```

Scores are cross-encoder logits (the unbounded reranker output), not probabilities. Used internally for ranking; the agent never sees them.

## Step 4: Wrap the pipeline as a retriever worker

The retriever-worker is a function that exposes the pipeline as a single tool the supervisor can call. It returns the structured envelope from the [retriever-as-worker concept page](../../concepts/multi-agent/retriever-as-worker.md#the-retriever-worker-contract).

Critical: the envelope carries `id` and `source` for each chunk. The supervisor must pass this envelope through to the synthesizer *verbatim* — the synthesizer cites by `id`. Any reformatting in the supervisor is where citation drift comes from.

In [ ]:
# Empirical floor for the cross-encoder score. Below this, chunks aren't
# usefully relevant. Tuned on the educational corpus; for production
# you'd tune per-corpus.
MIN_RERANK_SCORE = -2.0


def retriever_agent(query: str, top_k: int = 5) -> dict:
    """Run the retrieval pipeline. Return structured envelope.

    Status values:
    - "ok": at least one chunk above MIN_RERANK_SCORE
    - "empty": no chunks above the floor (corpus doesn't have it)
    - "error": pipeline failure

    The supervisor must not reason about per-chunk scores — they're
    informational only. If chunks are present, they're usable.
    """
    try:
        chunks = retrieve_chunks(query, top_k=top_k)
    except Exception as e:
        return {
            "status": "error", "kind": "pipeline_error",
            "detail": f"{type(e).__name__}: {e}",
            "chunks": [], "query": query, "pipeline": PIPELINE,
        }

    # Filter by relevance floor
    usable = [c for c in chunks if c["score"] >= MIN_RERANK_SCORE]
    if not usable:
        return {
            "status": "empty",
            "detail": ("no chunks above relevance floor "
                       f"(top score: {chunks[0]['score']:.2f} < {MIN_RERANK_SCORE})"),
            "chunks": [], "query": query, "pipeline": PIPELINE,
        }
    return {
        "status": "ok",
        "chunks": usable,
        "query": query,
        "pipeline": PIPELINE,
    }


# Try it
result = retriever_agent("how does ReAct differ from the basic agent loop?")
print(f"Status: {result['status']}, pipeline: {result['pipeline']}")
print(f"Chunks: {len(result['chunks'])}")
for c in result["chunks"][:2]:
    print(f"  [{c['id']}] {c['title']}: {c['text'][:100]}...")


## Step 5: The synthesizer worker

A Lab 10-style writer worker. System prompt requires inline `[chunk_id]` citations and renders a citation list at the end. The synthesizer reads the chunks envelope from the retriever — passed through by the supervisor verbatim — and composes the final answer.

The synthesizer is **not allowed to paraphrase claims into something the chunks don't support**. This is the mitigation for chunk drift. The system prompt has an explicit rule: if the chunks don't cover an aspect of the question, say so rather than inventing.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer agent. You receive a user
query plus a list of retrieved chunks from a document corpus.

Your job: produce a clear, accurate answer to the query using ONLY the
information in the chunks.

Rules:
1. CITE BY CHUNK_ID. For every claim drawn from a chunk, reference the
   chunk's id inline like [chunk_id_here]. Then list citations at the end:

       [chunk_id_1] source_filename — title
       [chunk_id_2] source_filename — title

2. DO NOT INVENT CLAIMS. If the chunks don't cover something the user asked,
   say so explicitly. Better to surface the gap than to paper over it.

3. DO NOT PARAPHRASE BEYOND WHAT THE CHUNKS SUPPORT. If a chunk says X, you
   can say X in different words. You cannot say "X implies Y" unless the
   chunk also said Y.

4. The user query may or may not have a corpus-grounded answer. If you can
   only address part of the question from the chunks, be explicit about
   which part.
"""


def synthesizer_agent(query: str, chunks: list[dict]) -> dict:
    """Compose the final answer from retrieved chunks."""
    if not chunks:
        # Should not happen if supervisor routes correctly, but defend.
        return {
            "status": "error", "kind": "no_chunks",
            "detail": "synthesizer called with empty chunks list",
        }

    chunks_block = "\n\n".join(
        f"[{c['id']}] (source: {c['source']}, title: {c['title']})\n{c['text']}"
        for c in chunks
    )

    user_prompt = (
        f"USER QUERY:\n{query}\n\n"
        f"RETRIEVED CHUNKS:\n{chunks_block}\n\n"
        "Compose the answer. Cite by chunk_id inline."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {
        "status": "ok",
        "answer": msg.content or "",
        # Track which chunks were available so the supervisor can verify
        # citation preservation downstream
        "chunk_ids_available": [c["id"] for c in chunks],
    }


## Step 6: The supervisor

Lab 10 pattern with two new worker tools: `call_retriever` and `call_synthesizer`. The system prompt encodes the four retrieval-decision rules from the [retriever-as-worker concept page](../../concepts/multi-agent/retriever-as-worker.md#the-four-retrieval-decision-rules) and includes a description of what the corpus covers — that's what lets the LLM decide whether a query is corpus-grounded.

`SUPERVISOR_MAX_STEPS = 6` is enough for the headline path (retrieve → synthesize → finalize is at most 3 supervisor calls). Higher cap would invite over-routing.

In [ ]:
SUPERVISOR_MAX_STEPS = 6


class CallRetrieverArgs(StrictModel):
    query: str = Field(
        description="A focused factual query. NOT the user's whole question; just "
                    "the specific claim/topic to retrieve."
    )
    top_k: int = Field(
        default=5,
        description="Number of chunks to retrieve (default 5; range 3-8 is sensible).",
    )


class CallSynthesizerArgs(StrictModel):
    query: str = Field(description="The user's original question.")
    chunks: list[dict] = Field(
        description=(
            "The chunks envelope from call_retriever, passed through VERBATIM. "
            "Do not re-format, filter, or summarize."
        ),
    )


def _call_retriever_tool(args: CallRetrieverArgs) -> dict:
    return retriever_agent(args.query, top_k=args.top_k)


def _call_synthesizer_tool(args: CallSynthesizerArgs) -> dict:
    return synthesizer_agent(args.query, args.chunks)


SUPERVISOR_TOOLS = {
    "call_retriever": (
        _call_retriever_tool, CallRetrieverArgs,
        "Search the document corpus for relevant chunks. Returns "
        "{status, chunks, query, pipeline}. Use this when the user's question "
        "asks about content in the corpus. Do NOT use for stable-knowledge "
        "questions the model already handles. Do NOT call more than once with "
        "the same query.",
    ),
    "call_synthesizer": (
        _call_synthesizer_tool, CallSynthesizerArgs,
        "Compose the final answer from retrieved chunks. Returns {status, answer}. "
        "Pass the chunks list from call_retriever EXACTLY as received — do not "
        "filter, re-number, or summarize. The synthesizer cites by chunk_id.",
    ),
}

CORPUS_DESCRIPTION = """The corpus covers core agentic AI concepts from Path 01:
the agent loop (how an agent iterates think → act → observe), tool design and
selection, the ReAct pattern (interleaved reasoning and action), the distinction
between search and retrieval, embeddings (what they are and how they're built),
vector indexes (HNSW, IVF, FAISS), chunking strategies, and citation tracking.

Each document is a markdown explainer ~200-500 words. The corpus does NOT cover:
multi-agent coordination, retrieval evaluation, language-specific tool SDKs,
or post-2024 framework updates.
"""

SUPERVISOR_SYSTEM_PROMPT = f"""You are a supervisor agent coordinating two workers:
a retriever (searches a document corpus) and a synthesizer (composes answers
from chunks).

CORPUS DESCRIPTION:
{CORPUS_DESCRIPTION}

YOUR JOB: For each user query, decide whether retrieval is needed. Then route.

RETRIEVAL DECISION RULES:

Rule 1 — RETRIEVE WHEN THE QUERY IS GROUNDED IN THE CORPUS. Compare the
query to the corpus description above. If the query asks about agent
loops, tools, ReAct, embeddings, vector indexes, chunking, or citations,
retrieve.

Rule 2 — DO NOT RETRIEVE FOR STABLE-KNOWLEDGE QUESTIONS. Definitional
questions, historical facts, general programming questions, math — these
don't need corpus retrieval. Answer directly.

Rule 3 — ONE RETRIEVAL PER DISTINCT FACTUAL QUESTION. If a retrieval
returns status='empty', surface "this isn't in the corpus" — do NOT
retry with reworded queries. The retrieval pipeline already does query
rewriting internally.

Rule 4 — PASS CHUNKS VERBATIM TO THE SYNTHESIZER. When you call
call_synthesizer, pass the chunks list from call_retriever's result
unchanged. Do not summarize, filter, or re-format chunks before passing
them.

WORKFLOW:
- If retrieval is needed: call_retriever → call_synthesizer (with the
  chunks) → return the synthesizer's answer.
- If retrieval is NOT needed: answer directly without calling either
  worker. Be honest that you're answering from training data, not the
  corpus.
- If the retriever returns status='empty': do NOT retry. Surface that
  the corpus doesn't cover the question; offer what you can from
  training data if appropriate (and say so).

CRITICAL: When the action-hash system refuses a repeated call, move on
instead of trying the same thing again.
"""


def _supervisor_make_schemas() -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        }
        for name, (_fn, args_model, description) in SUPERVISOR_TOOLS.items()
    ]


def _supervisor_execute_tool(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS:
        return {"status": "error", "kind": "unknown_worker",
                "tool": call.name, "available": list(SUPERVISOR_TOOLS)}
    fn, args_model, _ = SUPERVISOR_TOOLS[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"status": "error", "kind": "supervisor_dispatch_error",
                "detail": f"{type(e).__name__}: {e}"}


def multi_agent_rag(query: str, verbose: bool = True) -> dict:
    """Top-level entry point. Returns {answer, trace, retrieved_chunks, did_retrieve}."""
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]
    seen_actions: set[str] = set()
    trace: list[dict] = []
    schemas = _supervisor_make_schemas()
    retrieved_chunks: list[dict] = []
    did_retrieve = False

    for step in range(1, SUPERVISOR_MAX_STEPS + 1):
        if verbose:
            print(f"\n── Supervisor step {step} ──")

        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            # Supervisor returned an answer directly (skip path or after synthesizer)
            if verbose:
                print(f"  ◆ FINAL: {(msg.content or '')[:100]}...")
            return {
                "answer": msg.content or "",
                "trace": trace,
                "retrieved_chunks": retrieved_chunks,
                "did_retrieve": did_retrieve,
                "steps": step,
            }

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc.name} with these args."}
                if verbose:
                    print(f"  ✗ {tc.name}(...) [REPEATED]")
            else:
                seen_actions.add(ah)
                if verbose:
                    print(f"  → {tc.name}(query={tc.arguments.get('query', '?')[:60]}...)")
                tool_result = _supervisor_execute_tool(tc)
                if tc.name == "call_retriever":
                    did_retrieve = True
                    if tool_result.get("status") == "ok":
                        retrieved_chunks = tool_result.get("chunks", [])

                if verbose:
                    summary = tool_result.get("answer") or tool_result.get("status")
                    summary = str(summary)[:80]
                    print(f"  ← status={tool_result.get('status', '?')}: {summary}...")

            trace.append({"step": step, "tool": tc.name,
                          "result_status": tool_result.get("status")})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:6000]})

    return {
        "answer": "[supervisor hit step cap]",
        "trace": trace, "retrieved_chunks": retrieved_chunks,
        "did_retrieve": did_retrieve, "steps": SUPERVISOR_MAX_STEPS,
    }


## Step 7: Run end-to-end on a corpus-grounded query

A question the corpus covers. Expected trajectory:
1. Supervisor recognizes the query is corpus-grounded → calls `call_retriever`.
2. Retriever returns chunks.
3. Supervisor passes chunks verbatim to `call_synthesizer`.
4. Synthesizer composes answer with `[chunk_id]` citations.
5. Supervisor returns the answer.

Total: 3 supervisor steps; 2 LLM calls beyond the supervisor's own routing.

In [ ]:
query = "How does the ReAct pattern differ from a basic agent loop?"
result = multi_agent_rag(query)

print("\n" + "=" * 70)
print("FINAL ANSWER:")
print("=" * 70)
print(result["answer"])
print()
print(f"did_retrieve: {result['did_retrieve']}")
print(f"chunks retrieved: {len(result['retrieved_chunks'])}")
print(f"steps: {result['steps']}")
print("\nTrace:")
for t in result["trace"]:
    print(f"  step {t['step']}: {t['tool']:<18s} → {t['result_status']}")


**Sample output (LLM responses will vary; trajectory should be stable):**

```
── Supervisor step 1 ──
  → call_retriever(query=ReAct pattern vs basic agent loop...)
  ← status=ok: ok...

── Supervisor step 2 ──
  → call_synthesizer(query=How does the ReAct pattern differ from a basic agent loop?...)
  ← status=ok: The ReAct pattern extends the basic agent loop by adding explicit...

── Supervisor step 3 ──
  ◆ FINAL: The ReAct pattern extends the basic agent loop by adding explicit reasoning...

======================================================================
FINAL ANSWER:
======================================================================
The ReAct pattern extends the basic agent loop by adding explicit
reasoning between the act and observe steps [03-react-pattern_000]. In
a basic agent loop, the agent iterates think → act → observe with the
"think" step often implicit in the LLM's tool-call decision. ReAct
makes the reasoning explicit: at each iteration, the agent emits a
thought, then an action, then observes the result, then thinks again
[03-react-pattern_001]. The pattern improves interpretability and
debugging at the cost of more tokens per turn [03-react-pattern_001].

[03-react-pattern_000] 03-react-pattern.md — The ReAct Pattern
[03-react-pattern_001] 03-react-pattern.md — The ReAct Pattern

did_retrieve: True
chunks retrieved: 5
steps: 3

Trace:
  step 1: call_retriever     → ok
  step 2: call_synthesizer   → ok
```

Three supervisor steps; one retrieval; one synthesis; clean citations to specific chunk IDs.

## Step 8: The retrieve/skip diagnostic

The supervisor's most important decision: *should I retrieve?* Test both cases.

**Case A**: A query that's clearly off-corpus and stable-knowledge. The corpus covers agentic concepts; "what year was Marie Curie born?" is a definitional question the model handles from training. The supervisor should NOT retrieve.

**Case B**: A query that's clearly on-corpus. "How does chunking work?" — the corpus has a chunking-strategies doc. The supervisor SHOULD retrieve.

If your supervisor retrieves on Case A, your retrieve-skip-rule isn't firing — the prompt is too permissive. If it doesn't retrieve on Case B, the corpus description is too restrictive. Run both as a routine pre-deployment check.

In [ ]:
print("─── Case A: off-corpus stable-knowledge query ───")
result_a = multi_agent_rag("What year was Marie Curie born?", verbose=True)
print(f"\ndid_retrieve: {result_a['did_retrieve']}  (expected: False)")
print(f"answer (first 200 chars): {result_a['answer'][:200]}...")

print("\n\n─── Case B: corpus-grounded query ───")
result_b = multi_agent_rag("How does chunking work?", verbose=True)
print(f"\ndid_retrieve: {result_b['did_retrieve']}  (expected: True)")
print(f"chunks: {len(result_b['retrieved_chunks'])}")


**Sample output (Case A):**

```
── Supervisor step 1 ──
  ◆ FINAL: Marie Curie was born in 1867. (Answering from training data; this isn't a corpus question.)

did_retrieve: False  (expected: False)
```

**Sample output (Case B):**

```
── Supervisor step 1 ──
  → call_retriever(query=chunking strategies...)
  ← status=ok: ok...

── Supervisor step 2 ──
  → call_synthesizer(query=How does chunking work?...)
  ← status=ok: Chunking splits documents into smaller pieces...

did_retrieve: True  (expected: True)
chunks: 5
```

Both cases land the right routing. If yours don't, the diagnostic surfaces the bug before it pollutes real query traces.

## Step 9: Failure-mode walkthrough

The four multi-agent-RAG failure modes with Lab 13's mitigations:

### Failure mode 1: Citation drift

**Symptom**: synthesizer cites by `[1]`, `[2]` (auto-numbered) instead of by chunk_id, so citations don't trace back to specific chunks.

**Mitigation**: structural — the supervisor passes the chunks envelope through to the synthesizer verbatim (`CallSynthesizerArgs.chunks: list[dict]`); the synthesizer's prompt explicitly says "cite by chunk_id." The `call_synthesizer` tool's description tells the supervisor "pass chunks list EXACTLY as received."

Look at Step 7's output: citations are `[03-react-pattern_000]`, not `[1]`. That's the discipline working.

### Failure mode 2: Retrieval skip

**Symptom**: supervisor decides not to retrieve when it should have. Falls back on training data and gives a confidently-wrong answer.

**Mitigation**: the corpus description in the supervisor's system prompt + retrieval-decision Rule 1 ("retrieve when grounded in corpus"). Step 8's diagnostic surfaces this — if you ask a corpus-grounded question and `did_retrieve` is False, the prompt isn't routing correctly.

### Failure mode 3: Retrieval over-call

**Symptom**: supervisor retrieves on every query, including obviously off-corpus stable-knowledge ones.

**Mitigation**: Rule 2 in the supervisor's prompt ("don't retrieve for stable-knowledge questions"). Step 8 Case A verifies this directly. If `did_retrieve` is True for the Marie Curie question, the prompt is too permissive.

### Failure mode 4: Chunk drift

**Symptom**: synthesizer "paraphrases" claims into something the chunks don't actually say. Citations look right but claims attached to them aren't supported.

**Mitigation 1**: synthesizer's system prompt rule "do not paraphrase beyond what the chunks support."

**Mitigation 2** (Step 10, stretch): add Lab 11's critic on synthesis. The critic checks every claim against the chunks; ungrounded claims trigger revision.

Without the critic, chunk drift is the hardest failure mode to catch — it's only visible if you compare the answer to the chunks. With the critic, it's caught structurally. That's why high-stakes synthesis tasks add the critic.

## Step 10 (stretch 1): Compose with Lab 11's critic

Add the generator-critic refinement loop on top of the synthesizer. The critic reads `(chunks, draft)` and flags claims that aren't supported by any chunk. Bounded refinement with `MAX_REFINEMENT_CYCLES = 3`, same as Lab 11.

This is the **chunk drift** mitigation made structural. The synthesizer's "don't paraphrase beyond what chunks support" prompt rule is best-effort; the critic enforces it.

In [ ]:
MAX_REFINEMENT_CYCLES = 3

CRITIC_ISSUE_KINDS = [
    "unsupported_claim",   # claim in draft doesn't appear in any chunk
    "wrong_attribution",   # claim cites chunk_X but chunk_X doesn't support it
    "missing_citation",    # claim should cite a chunk but doesn't
]

CRITIC_SYSTEM_PROMPT = """You are a STRICT reviewer of a synthesizer's draft answer.

You receive a user query, the retrieved chunks (with chunk_ids), and the
synthesizer's draft.

Apply these checks IN ORDER:

1. unsupported_claim — for each factual claim in the draft, find a sentence
   in some chunk that supports it. If you cannot find supporting text in
   any chunk, flag it.

2. wrong_attribution — for each citation [chunk_id], check that the cited
   chunk actually supports the surrounding claim. If the chunk_id is real
   but the claim doesn't appear in that chunk, flag wrong_attribution.

3. missing_citation — claims that should cite a chunk but don't.

RULES:
- Default to OK on borderline cases. When uncertain, return ok.
- Quote the specific draft passage that fails the check.
- Cap at 3 issues. More than 3 → structural problems, not point fixes.
- Return ONLY valid JSON. No prose preamble. No markdown fences.

If draft passes all checks:
   {"status": "ok"}

If any check fails:
   {"status": "needs_revision", "issues": [{"kind": "<kind>", "detail": "<quote + explanation>"}, ...]}
"""


def critic_agent(query: str, chunks: list[dict], draft: str) -> dict:
    """Review the draft for groundedness against the chunks."""
    chunks_block = "\n\n".join(
        f"[{c['id']}]\n{c['text']}" for c in chunks
    )
    user_prompt = (
        f"USER QUERY:\n{query}\n\n"
        f"CHUNKS:\n{chunks_block}\n\n"
        f"DRAFT TO REVIEW:\n{draft}\n\n"
        "Apply the 3 checks. Return JSON only."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": CRITIC_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    raw = (msg.content or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        return {"status": "ok", "_parse_warning": raw[:200]}
    if result.get("status") not in ("ok", "needs_revision"):
        return {"status": "ok", "_invalid_status": result.get("status")}
    if result["status"] == "needs_revision":
        issues = result.get("issues", []) or []
        valid = [i for i in issues
                 if isinstance(i, dict) and i.get("kind") in CRITIC_ISSUE_KINDS][:3]
        return {"status": "needs_revision", "issues": valid}
    return {"status": "ok"}


def synthesize_with_critic(query: str, chunks: list[dict],
                            verbose: bool = True) -> dict:
    """Synthesizer with bounded critic-driven refinement."""
    draft_result = synthesizer_agent(query, chunks)
    if draft_result["status"] != "ok":
        return draft_result

    for cycle in range(MAX_REFINEMENT_CYCLES + 1):
        critic_result = critic_agent(query, chunks, draft_result["answer"])
        if verbose:
            print(f"  cycle {cycle}: critic status={critic_result['status']}")
            if critic_result["status"] == "needs_revision":
                for issue in critic_result.get("issues", []):
                    print(f"    [{issue['kind']}] {issue['detail'][:120]}...")

        if critic_result["status"] == "ok":
            return {
                "status": "ok", "answer": draft_result["answer"],
                "refinement_cycles": cycle, "final_critic_status": "ok",
            }
        if cycle == MAX_REFINEMENT_CYCLES:
            # Cap hit — surface honestly
            return {
                "status": "ok_with_unresolved_issues",
                "answer": draft_result["answer"],
                "refinement_cycles": cycle,
                "unresolved_issues": critic_result.get("issues", []),
            }

        # Revise with critic's issues
        issues_lines = [
            f"- [{i['kind']}] {i['detail']}"
            for i in critic_result.get("issues", [])
        ]
        revision_prompt = (
            "REVISION REQUESTED — address each of these critic issues:\n"
            + "\n".join(issues_lines)
            + f"\n\nOriginal query: {query}\n\n"
            "Original chunks:\n"
            + "\n\n".join(f"[{c['id']}]\n{c['text']}" for c in chunks)
            + f"\n\nPrevious draft:\n{draft_result['answer']}\n\n"
            "Produce a revised answer that addresses each issue."
        )
        msg = chat_with_tools(
            [{"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
             {"role": "user", "content": revision_prompt}],
            temperature=0,
        )
        draft_result = {"status": "ok", "answer": msg.content or ""}


# Run with the critic on a corpus query
print("─── Multi-agent RAG with critic-on-synthesis ───")
query = "What is the difference between dense retrieval and BM25?"
chunks_envelope = retriever_agent(query)
if chunks_envelope["status"] == "ok":
    result = synthesize_with_critic(query, chunks_envelope["chunks"])
    print(f"\nstatus: {result['status']}")
    print(f"refinement cycles: {result['refinement_cycles']}")
    print(f"\nFinal answer:\n{result['answer'][:500]}...")
else:
    print(f"Retrieval failed: {chunks_envelope['status']}")


On most queries the critic returns `ok` on the first draft — Lab 13's synthesizer prompt is already strict. The critic earns its place on edge cases: queries where the chunks have partial information and the synthesizer's first instinct is to fill the gap with training-data knowledge. The critic catches that "filling" as `unsupported_claim`.

Cost: roughly doubles the synthesis cost (critic + potential revision). Worth it for high-stakes synthesis; overhead for chat-style FAQ use cases.

## Step 11 (stretch 2): Compose with Lab 12's planner

For compound queries ("compare X, Y, Z"), Lab 12's planner emits a Plan with multiple `retrieve` steps in the same `parallel_group`. Executors run them concurrently via `ThreadPoolExecutor`; the synthesizer merges across all chunks.

This is a *light* integration — we don't replicate all of Lab 12's machinery (full Plan/PlanStep schemas, replanning, etc.). We just show the integration shape: parallel retrievals via the pool, merged synthesis. For the full pattern, see Lab 12.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

MAX_PARALLEL_RETRIEVALS = 3


def parallel_retrieve_and_synthesize(query: str, sub_queries: list[str],
                                       verbose: bool = True) -> dict:
    """Run multiple retrievals in parallel; merge chunks; synthesize.

    For compound queries that decompose into independent retrievals.
    sub_queries is the planner's output (here, hand-crafted; in Lab 12
    the planner emits these structurally).
    """
    if verbose:
        print(f"Original query: {query}")
        print(f"Decomposed into {len(sub_queries)} parallel retrievals:")
        for sq in sub_queries:
            print(f"  - {sq}")

    # Run retrievals in parallel
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_RETRIEVALS) as pool:
        results = list(pool.map(lambda sq: retriever_agent(sq, top_k=3), sub_queries))

    # Merge chunks; deduplicate by chunk_id
    seen_ids: set[str] = set()
    merged_chunks: list[dict] = []
    for r in results:
        if r["status"] != "ok":
            continue
        for c in r["chunks"]:
            if c["id"] not in seen_ids:
                seen_ids.add(c["id"])
                merged_chunks.append(c)

    if verbose:
        print(f"\nMerged {len(merged_chunks)} unique chunks from {len(sub_queries)} retrievals")

    if not merged_chunks:
        return {"status": "empty", "answer": "No chunks retrieved across any sub-query."}

    # Synthesize across all chunks
    result = synthesizer_agent(query, merged_chunks)
    return {
        "status": result["status"],
        "answer": result["answer"],
        "merged_chunk_count": len(merged_chunks),
        "sub_queries": sub_queries,
    }


# Compound query: 3 sub-questions that retrieve independently
compound_query = "Compare embeddings, BM25, and vector indexes."
sub_queries = [
    "what are embeddings",
    "how does BM25 work",
    "what is a vector index",
]
result = parallel_retrieve_and_synthesize(compound_query, sub_queries)
print(f"\nFinal answer:\n{result['answer'][:600]}...")
print(f"\nMerged chunk count: {result['merged_chunk_count']}")


Wall-clock: 3 retrievals running concurrently take ~1-2 seconds (cross-encoder is the bottleneck; pool gives you the embedding/BM25 parallelism). Sequentially they'd take ~3-6 seconds.

In a full Lab 12 integration, the planner would emit the `sub_queries` structurally (as `PlanStep` objects with `parallel_group="retrievals"`), and the supervisor would orchestrate the full plan. The pattern composes naturally — Lab 12's planner-executor + Lab 13's retriever-as-worker is `retrieve` becoming one of the planner's available tools.

## What you just built

A multi-agent RAG system in ~500 lines of Python total, integrating Lab 06-08's retrieval pipeline with Lab 10's coordination pattern:

- A retriever-worker that wraps the full Lab 07 v2 pipeline (dense + BM25 + RRF + cross-encoder rerank), auto-upgrading to v3 (Lab 08's contextual augmentation) when the cache is available.
- A synthesizer worker that cites by chunk_id and refuses to paraphrase beyond what chunks support.
- A supervisor that decides when to retrieve using the four retrieval-decision rules, with a corpus description in its system prompt that lets the LLM judge whether queries are corpus-grounded.
- A retrieve/skip diagnostic to verify the supervisor's routing before deployment.
- The four multi-agent-RAG failure modes (citation drift, retrieval skip, retrieval over-call, chunk drift) with structural mitigations for each.
- Optional composition with Lab 11's critic for groundedness (stretch 1).
- Optional composition with Lab 12's planner for parallel retrievals on compound queries (stretch 2).

The integration is pure composition: Lab 06-08's retrieval pipeline is **used unchanged**; Lab 10's supervisor pattern is **extended in place** with new worker tools; Lab 11's critic and Lab 12's planner **slot in** as optional layers without modifying anything else. That's the bet of Path 03's from-scratch discipline — the patterns compose because the contracts between them are explicit (structured envelopes, action-hash dedup, step caps that escalate via structured errors).

## Production readiness — out of scope here

For a real deployment you'd also want: vector DB integration (Qdrant/Pinecone/Weaviate) instead of in-memory NumPy + BM25; sharded retrieval across multiple corpora; per-query budget enforcement (latency, cost, max chunks returned); structured logging at every handoff; circuit breakers when the retriever consistently returns `empty`; eval-time monitoring of retrieve-skip-rate (a sudden spike means your supervisor's routing prompt has drifted); plan-quality eval on the Lab 12 stretch (how often does the planner emit non-decomposable plans for non-decomposable queries?).

## Next

- Take the [multi-agent RAG quiz](../../quizzes/multi-agent/multi-agent-rag.md).
- Path 03 continues with Module 5 (framework bridge — re-implementing Labs 10-13 in LangGraph's multi-agent primitives) in a future batch.
- A follow-up solutions batch will provide reference implementations for Labs 10/11/12/13 together.